# 가장 푸른 영상은 하루의 모습인가?

NDVI 최대 합성의 픽셀 선택 원리 확인

**할 일** · NDVI 최대 합성과 픽셀별 관측일을 나란히 확인

위에서 아래로 실행. 코드를 바꾸기 전 결과를 예상하고, 실행 결과와 비교

## 실행 준비

CPU 런타임 사용. Earth Engine에 등록한 Cloud 프로젝트와 사용 권한 필요. [설정 안내](https://github.com/steelpl/lecture/blob/main/CE776-climate-change-carbon-water-cycle/GEE_SETUP.md) 참고. 데이터 조회에는 본인 계정 인증 필요

In [ ]:
%pip -q install earthengine-api==1.7.52 geemap==0.39.0

In [ ]:
import ee
import geemap

PROJECT_ID = ""  # Earth Engine에 등록한 본인의 Cloud 프로젝트 ID
if not PROJECT_ID.strip():
    raise ValueError("PROJECT_ID에 본인의 등록된 Cloud 프로젝트 ID 입력")
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

In [ ]:
roi = ee.Geometry.Rectangle([126.80, 37.43, 127.15, 37.70], geodesic=False)

In [ ]:
def mask_s2(image):
    scl = image.select('SCL')
    clear = (scl.neq(1).And(scl.neq(3)).And(scl.neq(8))
             .And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11)))
    return (image.select(['B2', 'B3', 'B4', 'B8', 'B11'])
            .multiply(0.0001).updateMask(clear)
            .copyProperties(image, ['system:time_start']))
collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
              .filterBounds(roi).filterDate('2020-05-01', '2020-10-01')
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60))
              .map(mask_s2))

def add_quality(image):
    ndvi = image.expression('(n-r)/(n+r)', {'n':image.select('B8'),'r':image.select('B4')}).rename('NDVI')
    day = ee.Date(image.get('system:time_start')).difference(ee.Date('2020-01-01'), 'day')
    date_band = ee.Image.constant(day).rename('day_since_jan1').toFloat()
    return image.addBands([ndvi, date_band])
quality = collection.map(add_quality)
greenest = quality.qualityMosaic('NDVI').clip(roi)
median_ndvi = quality.select('NDVI').median().clip(roi)

In [ ]:
print('scenes:', collection.size().getInfo())
print('NDVI mean:', greenest.select('NDVI').reduceRegion(ee.Reducer.mean(),roi,30,maxPixels=2_000_000).getInfo())

In [ ]:
Map = geemap.Map()
Map.centerObject(roi, 10)
Map.addLayer(greenest.select('NDVI'), {'min':0,'max':1,'palette':['white','green']}, 'Max NDVI')
Map.addLayer(median_ndvi, {'min':0,'max':1,'palette':['white','green']}, 'Median NDVI')
Map.addLayer(greenest.select('day_since_jan1'), {'min':121,'max':273,'palette':['blue','yellow','red']}, 'Selected day (0 = Jan 1)')
Map

픽셀별 선택일이 모두 같은지 확인하시오. NDVI 최대 합성을 한 번의 위성 촬영으로 해석할 때 생기는 문제를 설명하시오. 수면·구름 잔여물이 큰 NDVI로 선택될 가능성도 검토하시오.

**정리** · NDVI는 식생 관련 지수. 탄소 흡수량이나 증발산량 자체가 아니며 별도 모형·검증 필요

## 자료와 해석 범위

- [Sentinel-2 SR](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED)

관측 연도는 고정 사례의 기간. 수업 연도와 무관. 코드는 자료 처리 방법을 보여 주며 지도·수치는 인증 후 조회 결과로 판단